# Egyptian Licence Plate — train on real Egypt data

Produces two things for the ITS pipeline:

1. **Plate colour classifier** trained on real Egyptian plates — the Egyptian
   plate band encodes vehicle CATEGORY (red = truck, light blue = private,
   orange = taxi), so this is a real signal for vehicle classification.
2. **A CSV** of every vehicle: plate colour, characters, and context.

### Read this before running

The **character/OCR** model does not need training — `sshdopey/egyptian-license-plates`
already publishes Egyptian ALPR models with 30+ Arabic character classes. This
notebook downloads and uses them.

What genuinely needs fitting is the **colour classifier**, because it is currently
tuned on a handful of hand-measured plates. That is what we train here, and it is
the part that works on the project's actual 720p footage.

> On `street_egypt.mp4`, plates are ~34px wide. OCR needs ~100px. Running the real
> Egyptian ALPR model on that clip produced 43 character detections, **none** of
> which were inside a plate — pure hallucination. Section 6 reproduces that test.
> Training does not fix it; it is a camera limit. See `docs/anpr-plan.md`.

## 1. GPU

In [ ]:
import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
else: print('Set Runtime > Change runtime type > T4 GPU')

## 2. Repo + dependencies

In [ ]:
import os, subprocess
REPO='https://github.com/yousseffbassemm/ITS-project-Elsewedy.git'
DIR='/content/its-traffic'
if os.path.isdir(DIR+'/.git'):
    subprocess.run(['git','-C',DIR,'pull','--ff-only'])
else:
    subprocess.run(['git','clone','-b','plates-anpr',REPO,DIR],check=True)
os.chdir(DIR); print(subprocess.run(['git','log','--oneline','-1'],capture_output=True,text=True).stdout)

In [ ]:
!pip install -q ultralytics supervision
import ultralytics; ultralytics.checks()

## 3. Get the pretrained Egyptian ALPR models

These do plate detection **and** Arabic character recognition in one pass.
No training needed — they are already trained on Egyptian plates.

In [ ]:
import urllib.request, pathlib
pathlib.Path('models').mkdir(exist_ok=True)

MODELS = {
  'models/plate_detect.pt':
    'https://huggingface.co/morsetechlab/yolov11-license-plate-detection/resolve/main/license-plate-finetune-v1n.pt',
  'models/eg_alpr.pt':
    'https://huggingface.co/sshdopey/egyptian-license-plates/resolve/main/license_yolo_med_97.pt',
}
for dest,url in MODELS.items():
    if not pathlib.Path(dest).exists():
        urllib.request.urlretrieve(url,dest)
    print(dest, round(pathlib.Path(dest).stat().st_size/1e6,1),'MB')

from ultralytics import YOLO
alpr = YOLO('models/eg_alpr.pt')
print('\nArabic character classes:', len(alpr.names))
print(sorted(set(alpr.names.values()))[:35])

## 4. Get real Egyptian plate images

Pick a source. Roboflow needs a free API key — put it in Colab **Secrets**
(key icon, left sidebar) as `ROBOFLOW_API_KEY`, never pasted in a cell.

Search Roboflow Universe for *egyptian license plate* and copy the
workspace / project / version from the dataset page into the cell below.

In [ ]:
from google.colab import userdata
import os
try:
    os.environ['ROBOFLOW_API_KEY']=userdata.get('ROBOFLOW_API_KEY'); print('key loaded')
except Exception as e:
    print('no key set:',e)

In [ ]:
!pip install -q roboflow
from roboflow import Roboflow
import os
rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
# <<< EDIT THESE THREE to the dataset you picked >>>
ds = (rf.workspace('yousef-gamal').project('egypt-car-plate')
        .version(4).download('yolov8', location='data/plates/egypt'))
print(ds.location)

In [ ]:
# What actually landed?
import pathlib
root=pathlib.Path('data/plates/egypt')
imgs=sorted(root.rglob('*.jpg'))+sorted(root.rglob('*.png'))
print(len(imgs),'images')
print([str(p.relative_to(root)) for p in imgs[:5]])

## 5. Build a plate-colour training set

Same bootstrap pattern the repo already uses in `tools/harvest_dataset.py`:
the model draws the boxes, the current rule drafts a label, and a human only
**corrects** — which is where nearly all annotation cost lives.

Each crop is split into the coloured band and the white body. The body is the
reference: judging the band against it cancels codec, white balance and exposure
together, which is what makes low-saturation CCTV plates readable at all.

In [ ]:
import cv2, numpy as np, pathlib, csv, sys
sys.path.insert(0,'/content/its-traffic')
from pipeline.plates import classify_band
from ultralytics import YOLO

det = YOLO('models/plate_detect.pt')
out = pathlib.Path('data/plates/color'); out.mkdir(parents=True,exist_ok=True)
(out/'crops').mkdir(exist_ok=True)

rows=[]
for i,p in enumerate(imgs):
    im=cv2.imread(str(p))
    if im is None: continue
    r=det.predict(im,imgsz=1280,conf=0.3,verbose=False)[0]
    for j,b in enumerate(r.boxes.xyxy.cpu().numpy()):
        x1,y1,x2,y2=[int(v) for v in b]
        if x2-x1 < 40: continue          # too small to label by eye
        plate=im[max(y1,0):y2, max(x1,0):x2]
        if plate.size==0: continue
        ph=plate.shape[0]; sp=max(ph//3,1)
        band,body=plate[:sp,:],plate[sp:,:]
        if band.size==0 or body.size==0: continue
        hb=cv2.cvtColor(band,cv2.COLOR_BGR2HSV); bo=cv2.cvtColor(body,cv2.COLOR_BGR2HSV)
        h,s,v=(float(np.median(hb[...,k])) for k in range(3))
        sb,vb=(float(np.median(bo[...,k])) for k in (1,2))
        draft=classify_band(h,s,v,sb,vb)
        name=f'{p.stem}_{j}.png'
        cv2.imwrite(str(out/'crops'/name), cv2.resize(plate,(160,80)))
        rows.append(dict(file=name,H=round(h),S=round(s),V=round(v),
                         body_S=round(sb),body_V=round(vb),
                         draft=draft,label=''))
with open(out/'labels.csv','w',newline='',encoding='utf-8-sig') as fh:
    w=csv.DictWriter(fh,fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
from collections import Counter
print(len(rows),'plate crops ->',out/'labels.csv')
print('draft colour mix:',Counter(r['draft'] for r in rows))

### Correct the labels

Open `data/plates/color/labels.csv`, look at the matching crop in `crops/`, and
fill the **`label`** column with one of:

`light_blue` (private) · `red` (truck) · `orange` (taxi) · `brown` (commercial) ·
`dark_blue` (police) · `green` (diplomatic) · `yellow` (customs)

Aim for **≥ 50 per colour**. Prioritise **light_blue vs dark_blue** — they differ
mainly in brightness, which small plates lose first, and that is the confusion the
current rule actually gets wrong on real footage.

In [ ]:
# Preview a grid so you can label quickly.
import matplotlib.pyplot as plt, cv2, pathlib
crops=sorted((out/'crops').glob('*.png'))[:24]
fig,ax=plt.subplots(4,6,figsize=(16,6))
for a,c in zip(ax.ravel(),crops):
    a.imshow(cv2.cvtColor(cv2.imread(str(c)),cv2.COLOR_BGR2RGB)); a.set_title(c.name[:14],fontsize=7); a.axis('off')
for a in ax.ravel()[len(crops):]: a.axis('off')
plt.tight_layout(); plt.show()

## 6. Fit the colour rule on the corrected labels

A decision tree on the six measured features, not a CNN. With a few hundred
samples a tree is the right capacity, and — more importantly — it stays
**inspectable**: you can read the thresholds it picked and check they are
physically sensible, which you cannot do with a black box.

In [ ]:
import pandas as pd, numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

df=pd.read_csv('data/plates/color/labels.csv')
df=df[df['label'].notna() & (df['label']!='')]
print(len(df),'labelled;',df['label'].value_counts().to_dict())
assert len(df)>=40,'label more crops first'

F=['H','S','V','body_S','body_V']
df['S_margin']=df['S']-df['body_S']       # the feature that actually matters
F=F+['S_margin']
X,y=df[F].values, df['label'].values
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.25,random_state=0,stratify=y)
clf=DecisionTreeClassifier(max_depth=5,min_samples_leaf=5,random_state=0).fit(Xtr,ytr)
print(classification_report(yte,clf.predict(Xte)))
print(confusion_matrix(yte,clf.predict(Xte)))
print(export_text(clf,feature_names=F))

In [ ]:
import joblib, json
joblib.dump(clf,'models/plate_color.joblib')
json.dump({'features':F,'classes':sorted(set(y))},open('models/plate_color_meta.json','w'),indent=2)
print('saved models/plate_color.joblib')

## 7. The reality check — reproduce the OCR finding

Run the real Egyptian ALPR model on the target clip. Upload `street_egypt.mp4`
first (it is gitignored — real CCTV of identifiable vehicles).

Expected: plates found at ~30px, and every 'character' a false positive sitting
outside any plate. That is the measurement behind the whole plan.

In [ ]:
import pathlib, os
clip=pathlib.Path('samples/street_egypt.mp4'); clip.parent.mkdir(exist_ok=True)
if not clip.exists():
    from google.colab import files
    up=files.upload()
    for n in up: os.replace(n,clip)
print(clip,'ok')

In [ ]:
import cv2, numpy as np
from ultralytics import YOLO
m=YOLO('models/eg_alpr.pt'); names=m.names
PLATE=[k for k,v in names.items() if v=='License Plate'][0]
CAR=[k for k,v in names.items() if v=='car'][0]
cap=cv2.VideoCapture('samples/street_egypt.mp4')
pw=[]; inside=0; outside=0; i=0
while i<900:
    ok,f=cap.read()
    if not ok: break
    if i%15==0:
        r=m.predict(f,imgsz=1280,conf=0.25,verbose=False)[0]
        bx=r.boxes.xyxy.cpu().numpy(); cl=r.boxes.cls.cpu().numpy().astype(int)
        pls=[b for b,c in zip(bx,cl) if c==PLATE]
        pw += [float(b[2]-b[0]) for b in pls]
        for b,c in zip(bx,cl):
            if c in (PLATE,CAR): continue
            cx,cy=(b[0]+b[2])/2,(b[1]+b[3])/2
            if any(p[0]<=cx<=p[2] and p[1]<=cy<=p[3] for p in pls): inside+=1
            else: outside+=1
    i+=1
cap.release()
print(f'plates: {len(pw)}  median width {np.median(pw):.0f}px' if pw else 'no plates')
print(f'characters INSIDE a plate : {inside}')
print(f'characters floating loose : {outside}   <- false positives')
print('\nOCR needs ~100px. This is a camera limit, not a training limit.')

## 8. Produce the CSV

One row per vehicle: colour, category (English + Arabic), characters, and — for
every empty character cell — **why** it is empty.

In [ ]:
!python -m pipeline.process_video --input samples/street_egypt.mp4 \
    --output-dir data/jobs/colab --stride 6 --plates
!python -m tools.export_plates_csv data/jobs/colab

In [ ]:
import pandas as pd
df=pd.read_csv('data/jobs/colab/plates.csv')
display(df)
from google.colab import files
files.download('data/jobs/colab/plates.csv')

## 9. Bring the weights home

Download these into `models/` locally. Do not commit `.pt` / `.joblib` — they are
gitignored.

Plate crops and the source clip are **personal data** (Egypt PDPL 151/2020):
clear this notebook's outputs before sharing it.

In [ ]:
from google.colab import files
import pathlib
for n in ('plate_color.joblib','plate_color_meta.json','eg_alpr.pt'):
    p=pathlib.Path('models')/n
    if p.exists(): print(n, round(p.stat().st_size/1e6,2),'MB'); files.download(str(p))
    else: print(n,'missing')